#**WEB SCRAPPING**

To build the benchmark dataset (`FINAL_MASTER_100K.csv`), data was extracted from two major news outlets: **The Edge** and **The Sun**. The scraping pipeline was heavily optimized for speed, reliability, and data quality using modern asynchronous Python libraries.

Below is a breakdown of the pipeline and the engineering optimizations applied:

### 1. Asynchronous Extraction (High-Speed I/O)
* **Libraries Used:** `asyncio` and `aiohttp`.
* **Optimization:** Instead of making standard, sequential HTTP requests (which force the code to wait for each page to load), the scraper uses asynchronous I/O. This allows the script to request dozens of web pages simultaneously, reducing extraction time from hours to mere minutes.

### 2. Resilience and Rate Limiting
* **Concurrency Control (Semaphores):** We implemented `asyncio.Semaphore` limits (50 for The Edge, 10 for The Sun) to cap the maximum number of simultaneous requests. This prevents overwhelming the target servers and avoids IP bans.
* **Exponential Backoff:** If the server returns a `429 Too Many Requests` error, the scraper uses a retry loop with exponential backoff (`asyncio.sleep(2 ** attempt)`). This automatically pauses the scraper and gracefully retries later.
* **Memory Management:** For massive scrapes (like The Edge), data is written in batches directly to the CSV file (`append` mode) rather than holding hundreds of thousands of records in RAM.

### 3. Data Standardization & Cleaning
Data from different APIs inherently have different structures and formats. We applied immediate transformation during extraction:
* **Timestamp Normalization:** Converted UNIX milliseconds (The Edge) and ISO strings (The Sun) into a standardized, timezone-aware datetime string format (`D/M/YYYY H:M:S AM/PM`).
* **Deep HTML Cleaning:** Applied regular expressions (`re.sub`) and `html.unescape()` to strip raw HTML tags, decode entities (like `&amp;`), and normalize whitespace, ensuring high-quality plain text for the `title` and `summary` columns.

### 4. Integration and Deduplication
* **Merging Sources:** Pandas (`pd.concat`) was used to vertically merge the independent datasets into a unified dataframe.
* **Deduplication:** Dropped duplicate records based on the unique article `link` to ensure the final benchmark tests exactly 100,000 distinct rows.
* **Missing Value Imputation:** Handled any empty cells by filling them with `"N/A"`, resulting in a clean, production-ready dataset for the final performance benchmark.

#**1. The Edge News**

In [ ]:
import asyncio
import aiohttp
import csv
import os
import re
import time
import html
from datetime import UTC, datetime

# ==========================================
# --- CONFIGURATION ---
# ==========================================
CATEGORIES_TO_SCRAPE = [
    "corporate", "economy", "malaysia", "world", "politics", "court",
    "markets", "stocks", "bursa-results", "warrants", "derivatives",
    "cryptocurrency", "islamic-finance", "fund-management", "personal-wealth",
    "options", "city-country", "wealth", "digitaledge", "esg", "tech",
    "property", "real-estate", "sme", "enterprise", "entrepreneur", "management",
    "opinion", "lifestyle", "sport", "sports", "education", "automotive", "auto",
    "aviation", "logistics", "features", "arts", "culture", "travel",
    "the-edge-tv", "videos", "podcasts", "gallery", "news", "weekly"
]

LIMIT_PER_PAGE = 20
CONCURRENCY_LIMIT = 50
BATCH_SIZE = 100

# ==========================================
# --- HELPERS ---
# ==========================================

def convert_timestamp(ms):
    """
    Converts millisecond timestamp to: 26/8/2025 8:27:30 AM
    """
    if not ms or ms == 0: return "N/A"
    try:
        dt = datetime.fromtimestamp(ms / 1000, tz=UTC)
        # Manual construction to ensure no leading zeros on day/month/hour across OS
        date_part = f"{dt.day}/{dt.month}/{dt.year}"
        time_part = dt.strftime("%I:%M:%S %p").lstrip("0")
        return f"{date_part} {time_part}"
    except Exception:
        return "N/A"

def clean_text(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'<.*?>', '', text) # Remove HTML
    text = html.unescape(text)        # Decode &amp; etc
    return re.sub(r'\s+', ' ', text).strip()

async def fetch_articles(session, offset, category, limit, semaphore):
    url = f"https://theedgemalaysia.com/api/loadMoreCategories?offset={offset}&categories={category}&limit={limit}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://theedgemalaysia.com/"
    }

    async with semaphore:
        for attempt in range(3):
            try:
                async with session.get(url, headers=headers, timeout=15) as response:
                    if response.status == 200:
                        data = await response.json()
                        return data.get("results", [])
                    elif response.status == 429:
                        await asyncio.sleep(2 ** attempt)
            except Exception:
                await asyncio.sleep(2 ** attempt)
        return None

def save_to_csv(data, filename):
    # Added article_id as the FIRST field
    fieldnames = [
        "article_id", "category", "sub-category", "title", "author",
        "source", "summary", "link", "created date", "updated date"
    ]
    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(data)

# ==========================================
# --- MAIN SCRAPER ---
# ==========================================

async def main():
    output_filename = "theedge.csv"
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)

    seen_links = set()
    total_collected_overall = 0
    start_time = time.time()

    async with aiohttp.ClientSession() as session:

        for current_category in CATEGORIES_TO_SCRAPE:
            print(f"\n🚀 === STARTING NEW CATEGORY: {current_category.upper()} ===")

            max_pages_per_category = 2000
            category_exhausted = False

            for batch_start in range(0, max_pages_per_category, BATCH_SIZE):
                if category_exhausted:
                    break

                batch_end = batch_start + BATCH_SIZE
                print(f"⚡ Fetching Pages {batch_start + 1} to {batch_end} for '{current_category}'...")

                tasks = [fetch_articles(session, i * LIMIT_PER_PAGE, current_category, LIMIT_PER_PAGE, semaphore)
                         for i in range(batch_start, batch_end)]

                batch_results = await asyncio.gather(*tasks)
                all_articles = []

                for page_articles in batch_results:
                    if page_articles is None: continue
                    if len(page_articles) == 0:
                        category_exhausted = True
                        continue

                    for item in page_articles:
                        alias_path = item.get("alias", "")
                        full_link = f"https://theedgemalaysia.com/{alias_path}" if alias_path else ""

                        if full_link in seen_links or not full_link:
                            continue
                        seen_links.add(full_link)

                        # Mapping data and injecting the Unique ID (nid)
                        all_articles.append({
                            "article_id": f"EDGE_{item.get('nid', '00000')}", # UNIQUE ID
                            "category": item.get("category", ""),
                            "sub-category": item.get("flash", ""),
                            "title": clean_text(item.get("title", "")),
                            "author": item.get("author", ""),
                            "source": item.get("source", ""),
                            "summary": clean_text(item.get("summary", "")),
                            "link": full_link,
                            "created date": convert_timestamp(item.get("created", 0)),
                            "updated date": convert_timestamp(item.get("updated", 0))
                        })
                        total_collected_overall += 1

                if all_articles:
                    save_to_csv(all_articles, output_filename)

            print(f"🏁 Finished all available articles in '{current_category}'.")

    elapsed_time = time.time() - start_time
    print(f"\n✅ ALL CATEGORIES COMPLETE!")
    print(f"📈 Total Unique Records Collected: {total_collected_overall}")
    print(f"⏱️ Total Time: {elapsed_time:.2f} seconds.")

# Execute
await main()

#**2. The Sun**

In [ ]:
import asyncio
import aiohttp
import pandas as pd
import re
import os
import html
from datetime import datetime

# ==========================================
# --- CONFIGURATION ---
# ==========================================
SUN_API_BASE = "https://thesun.my/wp-json/wp/v2/posts"
PER_PAGE = 50
PAGES_TO_FETCH = 150
CONCURRENCY = 10

# ==========================================
# --- HELPERS: CLEANING & DATE FORMATTING ---
# ==========================================
def format_sun_date(iso_date_str):
    """Converts ISO format to: 26/8/2025 8:27:30 AM"""
    if not iso_date_str: return ""
    try:
        dt = datetime.fromisoformat(iso_date_str)
        # We manually format to ensure no leading zeros on Day, Month, and Hour
        # as %-d / %-m behavior varies by OS.
        date_part = f"{dt.day}/{dt.month}/{dt.year}"
        time_part = dt.strftime("%I:%M:%S %p").lstrip("0") # Format: 8:27:30 AM
        return f"{date_part} {time_part}"
    except Exception:
        return iso_date_str

def clean_html(text):
    """Removes HTML tags and decodes entities for high-quality data."""
    if not isinstance(text, str): return ""
    text = re.sub(r'<.*?>', '', text) # Strip tags
    text = html.unescape(text)        # Convert &amp; to & etc.
    return re.sub(r'\s+', ' ', text).strip()

# ==========================================
# --- STEP 1: ASYNC SCRAPER FOR THE SUN ---
# ==========================================
async def fetch_sun_page(session, page, semaphore):
    params = {"page": page, "per_page": PER_PAGE}
    async with semaphore:
        try:
            async with session.get(SUN_API_BASE, params=params, timeout=30) as resp:
                if resp.status == 200:
                    return await resp.json()
                else:
                    return []
        except Exception:
            return []

async def run_sun_scraper():
    semaphore = asyncio.Semaphore(CONCURRENCY)
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_sun_page(session, p, semaphore) for p in range(1, PAGES_TO_FETCH + 1)]
        results = await asyncio.gather(*tasks)

        all_posts = [item for sublist in results for item in sublist]

        sun_data = []
        for post in all_posts:
            # Constructing the dictionary with article_id as the FIRST key
            sun_data.append({
                "article_id": f"SUN_{post.get('id', '00000')}", # Unique ID from the source
                "category": "News",
                "sub-category": "General",
                "title": clean_html(post.get("title", {}).get("rendered", "")),
                "author": "The Sun Reporter",
                "source": "The Sun Daily",
                "summary": clean_html(post.get("excerpt", {}).get("rendered", "")),
                "link": post.get("link", ""),
                "created date": format_sun_date(post.get("date", "")),
                "updated date": format_sun_date(post.get("modified", ""))
            })

        return pd.DataFrame(sun_data)

# ==========================================
# --- EXECUTION ---
# ==========================================
print("🚀 Starting The Sun Scraping...")
sun_df = await run_sun_scraper()

# Final Polish: Remove any duplicates based on the link
sun_df.drop_duplicates(subset=['link'], keep='first', inplace=True)

print(f"✅ Collected {len(sun_df)} unique records from The Sun.")

# Preview the data to verify the ID and Date format
print("\n--- Data Preview ---")
print(sun_df[['article_id', 'created date', 'title']].head())

# Save for the next step (Combining with The Edge)
sun_df.to_csv("thesun.csv", index=False)

#**Combine Sources**

In [ ]:
if os.path.exists("theedge.csv"):
    edge_df = pd.read_csv("theedge.csv")
    print(f"📂 Loaded {len(edge_df)} records from The Edge.")

    # Merge both
    master_df = pd.concat([edge_df, sun_df], ignore_index=True)
else:
    print("❌ Warning: theedge.csv not found.")
    master_df = sun_df

📂 Loaded 93387 records from The Edge.


#**Cleaning**

In [ ]:
def clean_content(text):
    if not isinstance(text, str): return ""
    # 1. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # 2. Decode HTML entities (e.g., &amp; -> &, &quot; -> ")
    text = html.unescape(text)
    # 3. Cleanup whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("🧹 Cleaning data and removing duplicates...")

# Apply cleaning
master_df['title'] = master_df['title'].apply(clean_content)
master_df['summary'] = master_df['summary'].apply(clean_content)

# Remove duplicates based on Link
initial_len = len(master_df)
master_df.drop_duplicates(subset=['link'], keep='first', inplace=True)

# Final formatting
master_df.fillna("N/A", inplace=True)

print(f"✨ Cleaning complete!")
print(f"📊 Final Unique Records: {len(master_df)}")

# Save to CSV
master_df.to_csv("FINAL_MASTER_100K.csv", index=False)
print("💾 'FINAL_MASTER_100K.csv' is ready for benchmarking!")

🧹 Cleaning data and removing duplicates...
✨ Cleaning complete!
📊 Final Unique Records: 100887
💾 'FINAL_MASTER_100K.csv' is ready for benchmarking!
